# Daily Challenge: Stock Price Prediction with LSTM

In [4]:
!pip install kaggle

In [6]:
import json
import os

# Remplace par ton API Token complet que tu as copié
kaggle_token = {
    "username": "tomisha",
    "key": "KGAT_de9b14eb939cd0abe44a911e9dce22b9"
}

# Créer le dossier .kaggle
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

# Écrire le fichier kaggle.json
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_token, f)

# Mettre les bonnes permissions
!chmod 600 ~/.kaggle/kaggle.json

In [7]:
!kaggle datasets list

ref                                                                 title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
dmahajanbe23/bmw-global-automotive-sales                            BMW Global Automotive Sales                              55017  2026-02-22 18:18:38.170000           7927        155  1.0              
shree0910/ai-and-data-science-job-market-dataset-20202026           AI & Data Science Job Market Dataset (2020–2026)        188367  2026-03-09 07:40:43.393000           1539         42  1.0              
ssssws/chocolate-sales-dataset-2023-2024                            Chocolate Sales Dataset 2023 - 2024                   24420255  2026-03-07 04:58:02.387000           2019         45

In [8]:
!kaggle datasets download -d jacksoncrow/stock-market-dataset
!unzip stock-market-dataset.zip -d stock_data

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
  inflating: stock_data/stocks/CAPL.csv  
  inflating: stock_data/stocks/CAPR.csv  
  inflating: stock_data/stocks/CAR.csv  
  inflating: stock_data/stocks/CARA.csv  
  inflating: stock_data/stocks/CARE.csv  
  inflating: stock_data/stocks/CARG.csv  
  inflating: stock_data/stocks/CARO.csv  
  inflating: stock_data/stocks/CARR#.csv  
  inflating: stock_data/stocks/CARS.csv  
  inflating: stock_data/stocks/CARV.csv  
  inflating: stock_data/stocks/CASA.csv  
  inflating: stock_data/stocks/CASH.csv  
  inflating: stock_data/stocks/CASI.csv  
  inflating: stock_data/stocks/CASS.csv  
  inflating: stock_data/stocks/CASY.csv  
  inflating: stock_data/stocks/CAT.csv  
  inflating: stock_data/stocks/CATB.csv  
  inflating: stock_data/stocks/CATC.csv  
  inflating: stock_data/stocks/CATM.csv  
  inflating: stock_data/stocks/CATO.csv  
  inflating: stock_data/stocks/CATS.csv  
  inflating: stock_data/stocks/CATY.csv  

In [9]:
import os
os.listdir('stock_data')

['etfs', 'stocks', 'symbols_valid_meta.csv']

## Explorer et charger les données

In [10]:
import pandas as pd

# Exemple : charger un fichier CSV d'action
df = pd.read_csv('stock_data/stocks/AAPL.csv')  # adapter le nom
print(df.head())
print(df.columns)

         Date      Open      High       Low     Close  Adj Close     Volume
0  1980-12-12  0.513393  0.515625  0.513393  0.513393   0.406782  117258400
1  1980-12-15  0.488839  0.488839  0.486607  0.486607   0.385558   43971200
2  1980-12-16  0.453125  0.453125  0.450893  0.450893   0.357260   26432000
3  1980-12-17  0.462054  0.464286  0.462054  0.462054   0.366103   21610400
4  1980-12-18  0.475446  0.477679  0.475446  0.475446   0.376715   18362400
Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'], dtype='object')


## Prétraitement des données

In [11]:
df['Target'] = df['Close'].shift(-1)
df = df.dropna()

In [12]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df[['Open', 'High', 'Low', 'Close', 'Volume']])

In [13]:
import numpy as np

sequence_length = 10
X, y = [], []

for i in range(len(scaled_features) - sequence_length):
    X.append(scaled_features[i:i+sequence_length])
    y.append(scaled_features[i+sequence_length, 3])  # Close du jour suivant

X = np.array(X)
y = np.array(y)

## Préparer le dataset PyTorch

In [14]:
import torch
from torch.utils.data import Dataset, DataLoader

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Diviser en train/test
split = int(len(X) * 0.8)
train_dataset = StockDataset(X[:split], y[:split])
test_dataset = StockDataset(X[split:], y[split:])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Définir le modèle LSTM

In [15]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # dernier pas de temps
        out = self.dropout(out)
        out = self.fc(out)
        return out

model = LSTMModel(input_size=X.shape[2], hidden_size=50, num_layers=2, output_size=1)

## Entraîner le modèle

In [16]:
import torch.optim as optim

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):
    model.train()
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.squeeze(), targets)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 1/20, Loss: 0.0001
Epoch 2/20, Loss: 0.0000
Epoch 3/20, Loss: 0.0000
Epoch 4/20, Loss: 0.0000
Epoch 5/20, Loss: 0.0000
Epoch 6/20, Loss: 0.0000
Epoch 7/20, Loss: 0.0000
Epoch 8/20, Loss: 0.0000
Epoch 9/20, Loss: 0.0000
Epoch 10/20, Loss: 0.0000
Epoch 11/20, Loss: 0.0000
Epoch 12/20, Loss: 0.0000
Epoch 13/20, Loss: 0.0000
Epoch 14/20, Loss: 0.0001
Epoch 15/20, Loss: 0.0000
Epoch 16/20, Loss: 0.0000
Epoch 17/20, Loss: 0.0000
Epoch 18/20, Loss: 0.0000
Epoch 19/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0000


## Évaluer le modèle

In [17]:
from sklearn.metrics import r2_score

model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = model(inputs)
        predictions.extend(outputs.squeeze().numpy())
        actuals.extend(targets.numpy())

print("R² score:", r2_score(actuals, predictions))

R² score: 0.698012634096411


In [18]:
torch.save(model.state_dict(), 'lstm_stock_model.pth')
import joblib
joblib.dump(scaler, 'scaler.save')

['scaler.save']